In [0]:
import logging


In [0]:
display(dbutils.fs.ls("abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/crm/,crm/,0,1790406871000
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/,erp/,0,1790406866000


In [0]:
display(
    dbutils.fs.ls(
        "abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/crm/"
    )
)

path,name,size,modificationTime
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/crm/cust_info.csv,cust_info.csv,855395,1790407412000
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/crm/prd_info.csv,prd_info.csv,26934,1790407413000
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/crm/sales_details.csv,sales_details.csv,3588116,1790407412000


In [0]:
display(
    dbutils.fs.ls(
        "abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/"
    )
)

path,name,size,modificationTime
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/CUST_AZ12.csv,CUST_AZ12.csv,561549,1790407383000
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/LOC_A101.csv,LOC_A101.csv,402669,1790407382000
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/PX_CAT_G1V2.csv,PX_CAT_G1V2.csv,1169,1790407383000


In [0]:
display(
    dbutils.fs.ls(
        "abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/"
    )
)

path,name,size,modificationTime
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/CUST_AZ12.csv,CUST_AZ12.csv,561549,1790407383000
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/LOC_A101.csv,LOC_A101.csv,402669,1790407382000
abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/erp/PX_CAT_G1V2.csv,PX_CAT_G1V2.csv,1169,1790407383000


In [0]:
base_path = "abfss://sink-sys@sinkstorageacc.dfs.core.windows.net/"
INGESTION_CONFIGURATION = [
    {
        'file_Name' : "CUST_AZ12",
        'source' : base_path + "erp/CUST_AZ12.csv",
        'table' : "CUST_AZ12",
    },
    {
        'file_Name' : "LOC_A101",
        'source' : base_path + "erp/LOC_A101.csv",
        'table' : "LOC_A101",
    },
    {
        'file_Name' : "PX_CAT_G1V2",
        'source' : base_path + "erp/PX_CAT_G1V2.csv",
        'table' : "PX_CAT_G1V2",
    },
    {
        'file_Name' : "cust_info",
        'source' : base_path + "crm/cust_info.csv",
        'table' : "cust_info"
    },
    {
        'file_Name' : "prd_info",
        'source' : base_path + "crm/prd_info.csv",
        'table' : "prd_info"
    },
    {
        'file_Name' : "sales_details",
        'source' : base_path + "crm/sales_details.csv",
        'table' : "sales_details"
    }
]

In [0]:
import logging
from pyspark.sql import functions as F

logger = logging.getLogger("databricks")

for file in INGESTION_CONFIGURATION:
    try:
        logger.info(f"Loading {file['file_Name']} to {file['table']}")

        df = (
            spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(file["source"])
                .withColumn("_ingest_timestamp", F.current_timestamp())
                .withColumn("_source_file", F.lit(file["source"]))
        )


        df.write\
            .mode("overwrite")\
            .format("delta")\
            .saveAsTable(f"medallionsales.bronze.{file['table']}")


        logger.info(f"Successfully loaded {file['file_Name']}")

    except Exception as e:
        logger.error(
            f"Error loading {file['file_Name']}: {e}"
        )
        raise